In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

# Parse gff3 genome files
Gather data from different feature and merge them if they belong to the same gene.

In [2]:
filenames = {
    'U00096': '../../../datasets/genomics/reference_genome/U00096.gff3',
    'JX871397': '../../../datasets/genomics/reference_genome/JX871397.gff3'
}

In [3]:
gene_info = dict()

for sequence_name in ['JX871397', 'U00096']:

    print(f'\nParsing chromosome {sequence_name}.\n')    
    file = open(filenames[sequence_name])
    
    skiprows = 6

    bt = []

    prev_lstart = 0
    prev_lend = 0
    prev_ltype = 'None'
    
    gene_info_chrom = dict() # dict of dicts with info on each gene feature

    for line in file.readlines()[skiprows:]:
        
            if not line.split():
                continue
                
            splitline = line.split('\t')
            ltype = splitline[2] # less informative than 'gbkey'
            lstart = splitline[3]
            lend = splitline[4]
            lstrand = splitline[6]
            linfo = splitline[8].replace('\n', '')

            # Get info about the feature
            # If ltype is gene: gene, Dbxref, gene_biotype (eg protein_coding), gene_synonym
            # If ltype is CDS: extract Uniprot ID from Dbxref, extract product (or Note if available)
            dinfo = {i.split('=')[0]: i.split('=')[1] for i in linfo.split(';')}

            #####################################
            #### Feature already in database ####
            #####################################

            # There's already a feature with the exact same position
            if lstart == prev_lstart and lend == prev_lend:

                bt += [prev_ltype + ' then ' + ltype]

                # rep_origin, misc_feature and mobile_element don't have additional features
                # Additional features have no extra info on target_type

                # Which gene record to add info to?
                previous_gene_name = dinfo['gene'] if 'gene' in dinfo.keys() else dinfo['locus_tag']

                # Extract UNIPROT ref
                db_indices = dinfo['Dbxref'] if 'Dbxref' in dinfo.keys() else ''
                if 'Swiss-Prot' in db_indices:
                    gene_info_chrom[previous_gene_name]['uniprot'] = db_indices.split('Swiss-Prot:')[1].split(',')[0]

                # Replace "Added later" with the product
                gene_info_chrom[previous_gene_name]['product'] = dinfo['Note'] if 'Note' in dinfo.keys() else dinfo['product']

                # Add locus tag
                locus_tag = dinfo.get('locus_tag', None)
                previous_locus_tag = gene_info_chrom[previous_gene_name].get('locus_tag', None)
                if locus_tag != previous_locus_tag and previous_locus_tag:
                    print(locus_tag, previous_locus_tag)
                gene_info_chrom[previous_gene_name]['locus_tag'] = locus_tag

            ####################################
            #### Feature with new positions ####
            ####################################

            else: # New gene: extract gene_biotype as target_type

                # Get the type of target
                target_type = dinfo['gene_biotype'] if dinfo['gbkey'] == 'Gene' else dinfo['gbkey']

                # Origin of replication: no further feature
                if target_type == 'rep_origin':
                    target_type = target_type
                    gene_name = dinfo['Note']
                    db_indices = ''
                    synonyms = ''
                    product = 'Origin of replication'

                # Misc_feature: pseudogene or cryptic prophage, no further feature
                elif target_type == 'misc_feature':

                    # Pseudogenes
                    if 'gene' in dinfo.keys():
                        target_type == 'pseudogene'
                        gene_name = dinfo['gene']
                        db_indices = dinfo['Dbxref']
                        synonyms = ''
                        product = dinfo['Note']

                    # Cryptic prophages
                    else:
                        target_type == 'cryptic_prophage'
                        gene_name = dinfo['Note'].split(' ')[-1]
                        db_indices = ''
                        synonyms = ''
                        product = dinfo['Note']

                # Regular gene
                elif 'gene' in dinfo.keys():
                    target_type = target_type
                    gene_name = dinfo['gene']
                    db_indices = dinfo['Dbxref'] if 'Dbxref' in dinfo.keys() else ''
                    synonyms = dinfo['gene_synonym'] if 'gene_synonym' in dinfo.keys() else ''
                    product = 'Added later' # should come with the next feature

                # Mobile genetic element
                elif 'mobile_element_type' in dinfo.keys():                   
                    target_type = 'mobile_element'
                    gene_name = dinfo['mobile_element_type'].split(':')[-1]
                    db_indices = dinfo['Dbxref'] if 'Dbxref' in dinfo.keys() else ''
                    synonyms = ''  
                    product = dinfo['mobile_element_type']
                    
                # Phage genes
                elif 'gbkey' in dinfo.keys():
                    
                    if dinfo['gbkey'] == 'Gene':
                        target_type = dinfo['gene_biotype']
                        gene_name = dinfo['Name']
                        product = dinfo['Name']        
                    
                    elif 'Note' in dinfo.keys():
                        
                        gene_name = dinfo['Note']
                        product = dinfo['Note']
                        
                    db_indices = ''
                    synonyms = ''
                    
                else:
                    print(f'Unknown: {target_type}, {gene_name}')
                    crash

                # Extract ECOCYC ref
                if db_indices:
                    ecocyc = db_indices.split('ECOCYC:')[-1]
                else:
                    ecocyc = ''

                # Extract ECOCYC ref
                if db_indices:
                    ecocyc = db_indices.split('ECOCYC:')[-1]
                else:
                    ecocyc = ''

                # New position, but already known gene_name
                if gene_name in gene_info_chrom.keys():

                    # Pseudogenes split in two by MGEs
                    # First comes the ltype "pseudogene", full-length, contains dbxrefs and synonyms
                    # Then two CDS features, contain dinfo["product"]
                    # Then two sequence_features, no info

                    # Add synonym information (just for insI2)
                    if 'gene_synonym' in dinfo.keys():
                        gene_info_chrom[gene_name]['synonyms'] = gene_info_chrom[gene_name]['synonyms'] + ',' + dinfo['gene_synonym']

                    # Add product information
                    if 'product' in dinfo.keys():

                        # There's no product information yet
                        if gene_info_chrom[gene_name]['product'] == 'Added later':
                            gene_info_chrom[gene_name]['product'] = dinfo['product']
                            
                        # There's already conflicting product information: skip (the first instance has better quality information)
                        elif gene_info_chrom[gene_name]['product'] != dinfo['product']:
                            print('Conflict:', gene_name, '; ', gene_info_chrom[gene_name]['product'], '; ', dinfo['product'])
                            continue

                    # Add locus tag
                    locus_tag = dinfo.get('locus_tag', None)
                    previous_locus_tag = gene_info_chrom[gene_name].get('locus_tag', None)
                    if locus_tag != previous_locus_tag and previous_locus_tag:
                        print(gene_name, locus_tag, previous_locus_tag)
                    gene_info_chrom[gene_name]['locus_tag'] = locus_tag

                    # Ignore the features that include no info on products or synonyms

                # If the feature is really new, record in the dict
                else:

                    # Add locus tag
                    locus_tag = dinfo.get('locus_tag', None)
                    previous_locus_tag = gene_info_chrom.get(gene_name, {}).get('locus_tag', None)
                    if locus_tag != previous_locus_tag and previous_locus_tag:
                        print(gene_name, locus_tag, previous_locus_tag)

                    gene_info_chrom[gene_name] = {'gene_name': gene_name, 'chrom': sequence_name, 'start': lstart, 'end': lend, 'strand': lstrand, 'target_type': target_type, 'product': product, 'locus_tag': locus_tag, 'ecocyc': ecocyc, 'uniprot': '', 'synonyms': synonyms}

                    prev_lstart = lstart
                    prev_lend = lend
                    prev_ltype = ltype

    file.close()
    
    # Add "unknown" product info for the genes still missing it 
    for k in gene_info_chrom.keys():
        if gene_info_chrom[k]['product'] == 'Added later':
            #print(k, gene_info_chrom[k]['product'])
            gene_info_chrom[k]['product'] = 'unknown'
    
        if gene_info_chrom[k]['product'] == 'WARNING':
            print(k)

    # Dict for all chromosomes
    gene_info[sequence_name] = gene_info_chrom


Parsing chromosome JX871397.

IR2 None PHI80_14
putative rightward transcription terminator None PHI80_17
predicted LexA binding site None PHI80_25
pinL None PHI80_30

Parsing chromosome U00096.

Conflict: mrcB ;  peptidoglycan glycosyltransferase/peptidoglycan DD-transpeptidase MrcB ;  PBP1Bgamma
Conflict: dnaX ;  DNA polymerase III subunit tau ;  DNA polymerase III subunit gamma
Conflict: dnaX ;  DNA polymerase III subunit tau ;  DNA polymerase III subunit gamma
Conflict: copA ;  Cu(+) exporting P-type ATPase ;  soluble Cu(+) chaperone
Conflict: copA ;  Cu(+) exporting P-type ATPase ;  soluble Cu(+) chaperone
Conflict: cobB ;  protein-lysine deacetylase/desuccinylase/lipoamidase ;  CobB-S
insI2 b1404 b4708
Conflict: clpB ;  chaperone protein ClpB ;  ClpB80
Conflict: yibX ;  protein YibX ;  protein YibX-S
Conflict: istR ;  small RNA IstR-2 ;  small regulatory RNA IstR-1
Conflict: istR ;  small RNA IstR-2 ;  small regulatory RNA IstR-1
Conflict: mcrB ;  5-methylcytosine-specific restr

# Make a gene map dataframe

In [4]:
gene_map_ecoli = pd.DataFrame.from_dict(gene_info['U00096'], orient = 'index').reset_index(drop = True)
gene_map_phage = pd.DataFrame.from_dict(gene_info['JX871397'], orient = 'index').reset_index(drop = True)

duplicate = np.intersect1d(gene_map_ecoli.gene_name, gene_map_phage.gene_name)
print(duplicate)

gene_map_ecoli['gene_name'] = np.where(gene_map_ecoli.gene_name.isin(duplicate), gene_map_ecoli.gene_name + '', gene_map_ecoli.gene_name)
gene_map_phage['gene_name'] = np.where(gene_map_phage.gene_name.isin(duplicate), gene_map_phage.gene_name + '_2', gene_map_phage.gene_name)

gene_map = pd.concat([gene_map_ecoli, gene_map_phage])

['ninE']


### Make a TSV file

In [5]:
gene_map.to_csv('../../../datasets/genomics/reference_genome/gene_map.tsv', sep = '\t', index = False)